In [2]:
import csv
import requests
from datetime import datetime

In [3]:
def crmls_sold(file_year, file_month):

    # Ensuring dates account for December 1st to January 1st instead of trying to find a non-existent 13th month
    if file_month == 12:
        file_month_end = 1
        file_year_end = file_year + 1
    else:
        file_month_end = file_month + 1
        file_year_end = file_year

    
    # Define the API endpoint
    url = 'https://api-trestle.corelogic.com/trestle/odata/Property'

    # Get token from IDX Exchange secure proxy instead of exposing CoreLogic credentials
    auth_endpoint = 'https://idxexchange.com/internal-api/trestle_token.php?key=IDXEXCHANGE2026_CHANGE_THIS'

    response = requests.get(auth_endpoint)
    response.raise_for_status()

    # Parse the response to extract the token
    token = response.json().get('access_token')

    if token:
        # Define the headers with the token
        headers = {
            'Authorization': f'Bearer {token}'
        }

        # Define the parameters for the API request
        params = {
            '$select': 'BuyerAgentAOR, ListAgentAOR, Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN, OriginalListPrice, ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,BuyerOfficeName,CoListOfficeName,ListAgentFullName,CoListAgentFirstName,CoListAgentLastName,BuyerAgentMlsId,BuyerAgentFirstName,BuyerAgentLastName,FireplacesTotal,AssociationFeeFrequency,AboveGradeFinishedArea,ListingKeyNumeric,MLSAreaMajor,TaxAnnualAmount,CountyOrParish,MlsStatus,ElementarySchool,AttachedGarageYN,ParkingTotal,BuilderName,PropertySubType,LotSizeAcres,SubdivisionName,BuyerOfficeAOR,YearBuilt,StreetNumberNumeric,ListingId,BathroomsTotalInteger,City,TaxYear,BuildingAreaTotal,BedroomsTotal,ContractStatusChangeDate,ElementarySchoolDistrict,CoBuyerAgentFirstName,PurchaseContractDate,ListingContractDate,BelowGradeFinishedArea,BusinessType,StateOrProvince,CoveredSpaces,MiddleOrJuniorSchool,FireplaceYN,Stories,HighSchool,Levels,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName',
      
            '$filter': f"MlsStatus eq 'Closed' and CloseDate ge {datetime(file_year, file_month, 1).isoformat(timespec='milliseconds')}Z and CloseDate lt {datetime(file_year_end, file_month_end, 1).isoformat(timespec='milliseconds')}Z",

            '$top': 1000  # Extracting up to 1000 observations
        }

        # Send a GET request to the API endpoint with the token and parameters
        total_records = 0
        if file_month < 10:
            csv_file = f"CRMLSSold{file_year}0{file_month}.csv"
        else:
            csv_file = f"CRMLSSold{file_year}{file_month}.csv"

        with open(csv_file, mode='w', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=['BuyerAgentAOR','ListAgentAOR','Flooring','ViewYN', 'WaterfrontYN','BasementYN','PoolPrivateYN','OriginalListPrice','ListingKey', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric',  'ListingId', 'BathroomsTotalInteger', 'City',  'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate',  'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'PurchaseContractDate', 'ListingContractDate', 'BelowGradeFinishedArea', 'BusinessType',  'StateOrProvince', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'LotSizeDimensions', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict','OriginatingSystemName','OriginatingSystemSubName'])
            writer.writeheader()

            while True:
                response = requests.get(url, params=params, headers=headers)
                if response.status_code == 200:
                    data = response.json()
                    observations = data.get('value', [])
                    for observation in observations:
                        writer.writerow({
                            'BuyerAgentAOR': observation.get('BuyerAgentAOR', ''),
                            'ListAgentAOR': observation.get('ListAgentAOR', ''),
                            'Flooring': observation.get('Flooring', ''),
                            'ViewYN': observation.get('ViewYN', ''),         
                            'WaterfrontYN': observation.get('WaterfrontYN', ''),
                            'BasementYN': observation.get('BasementYN', ''),
                            'PoolPrivateYN': observation.get('PoolPrivateYN', ''),
                          'OriginalListPrice': observation.get('OriginalListPrice', ''),
                            'ListingKey': observation.get('ListingKey', ''),
                            'CloseDate': observation.get('CloseDate', ''),
                            'ClosePrice': observation.get('ClosePrice', ''),
                            'ListAgentFirstName': observation.get('ListAgentFirstName', ''),
                            'ListAgentLastName': observation.get('ListAgentLastName', ''),
                            'Latitude': observation.get('Latitude', ''),
                            'Longitude': observation.get('Longitude', ''),
                            'UnparsedAddress': observation.get('UnparsedAddress', ''),
                            'PropertyType': observation.get('PropertyType', ''),
                            'LivingArea': observation.get('LivingArea', ''),
                            'ListPrice': observation.get('ListPrice', ''),
                            'DaysOnMarket': observation.get('DaysOnMarket', ''),
                            'ListOfficeName': observation.get('ListOfficeName', ''),
                            'BuyerOfficeName': observation.get('BuyerOfficeName', ''),
                            'CoListOfficeName': observation.get('CoListOfficeName', ''),
                            'ListAgentFullName': observation.get('ListAgentFullName', ''),
                            'CoListAgentFirstName': observation.get('CoListAgentFirstName', ''),
                            'CoListAgentLastName': observation.get('CoListAgentLastName', ''),
                            'BuyerAgentMlsId': observation.get('BuyerAgentMlsId', ''),
                            'BuyerAgentFirstName': observation.get('BuyerAgentFirstName', ''),
                            'BuyerAgentLastName': observation.get('BuyerAgentLastName', ''),
                            'FireplacesTotal': observation.get('FireplacesTotal', ''),
                            'AssociationFeeFrequency': observation.get('AssociationFeeFrequency', ''),
                            'AboveGradeFinishedArea': observation.get('AboveGradeFinishedArea', ''),
                            'ListingKeyNumeric': observation.get('ListingKeyNumeric', ''),
                            'MLSAreaMajor': observation.get('MLSAreaMajor', ''),
                            'TaxAnnualAmount': observation.get('TaxAnnualAmount', ''),
                            'CountyOrParish': observation.get('CountyOrParish', ''),
                            'MlsStatus': observation.get('MlsStatus', ''),
                            'ElementarySchool': observation.get('ElementarySchool', ''),
                            'AttachedGarageYN': observation.get('AttachedGarageYN', ''),
                            'ParkingTotal': observation.get('ParkingTotal', ''),
                            'BuilderName': observation.get('BuilderName', ''),
                            'PropertySubType': observation.get('PropertySubType', ''),
                            'LotSizeAcres': observation.get('LotSizeAcres', ''),
                            'SubdivisionName': observation.get('SubdivisionName', ''),
                            'BuyerOfficeAOR': observation.get('BuyerOfficeAOR', ''),
                            'YearBuilt': observation.get('YearBuilt', ''),
                            'StreetNumberNumeric': observation.get('StreetNumberNumeric', ''),                     
                            'ListingId': observation.get('ListingId', ''),
                            'BathroomsTotalInteger': observation.get('BathroomsTotalInteger', ''),
                            'City': observation.get('City', ''),
                            'TaxYear': observation.get('TaxYear', ''),
                            'BuildingAreaTotal': observation.get('BuildingAreaTotal', ''),
                            'BedroomsTotal': observation.get('BedroomsTotal', ''),
                            'ContractStatusChangeDate': observation.get('ContractStatusChangeDate', ''),                      
                            'ElementarySchoolDistrict': observation.get('ElementarySchoolDistrict', ''),
                            'CoBuyerAgentFirstName': observation.get('CoBuyerAgentFirstName', ''),
                            'PurchaseContractDate': observation.get('PurchaseContractDate', ''),
                            'ListingContractDate': observation.get('ListingContractDate', ''),
                            'BelowGradeFinishedArea': observation.get('BelowGradeFinishedArea', ''),
                            'BusinessType': observation.get('BusinessType', ''),                                            
                            'StateOrProvince': observation.get('StateOrProvince', ''),
                            'CoveredSpaces': observation.get('CoveredSpaces', ''),
                            'MiddleOrJuniorSchool': observation.get('MiddleOrJuniorSchool', ''),
                            'FireplaceYN': observation.get('FireplaceYN', ''),
                            'Stories': observation.get('Stories', ''),
                            'HighSchool': observation.get('HighSchool', ''),
                            'Levels': observation.get('Levels', ''),                 
                            'LotSizeDimensions': observation.get('LotSizeDimensions', ''),
                            'LotSizeArea': observation.get('LotSizeArea', ''),
                            'MainLevelBedrooms': observation.get('MainLevelBedrooms', ''),
                            'NewConstructionYN': observation.get('NewConstructionYN', ''),
                            'GarageSpaces': observation.get('GarageSpaces', ''),
                            'HighSchoolDistrict': observation.get('HighSchoolDistrict', ''),
                            'PostalCode': observation.get('PostalCode', ''),
                            'AssociationFee': observation.get('AssociationFee', ''),
                            'LotSizeSquareFeet': observation.get('LotSizeSquareFeet', ''),
                            'MiddleOrJuniorSchoolDistrict': observation.get('MiddleOrJuniorSchoolDistrict', ''),
                            'OriginatingSystemName': observation.get('OriginatingSystemName', ''),
                            'OriginatingSystemSubName': observation.get('OriginatingSystemSubName', ''),
                       
                        })
                        total_records += 1

                    # Check if there are more records to fetch
                    if '@odata.nextLink' in data:
                        next_link = data['@odata.nextLink']
                        params = None  # Clear params to avoid appending to the existing query string
                        url = next_link
                    else:
                        break
                else:
                    print(f"Error: {response.status_code}")
                    print(f"Error Message: {response.text}")
                    break

        print(f"Total {total_records} records exported to {csv_file}")
    else:
        print("Error retrieving token: access_token not found")


In [4]:
def crmls_listed(file_year, file_month):

    # Ensuring dates account for December 1st to January 1st instead of trying to find a non-existent 13th month
    if file_month == 12:
        file_month_end = 1
        file_year_end = file_year + 1
    else:
        file_month_end = file_month + 1
        file_year_end = file_year


    # Define the API endpoint
    url = 'https://api-trestle.corelogic.com/trestle/odata/Property'

    # Get token from IDX Exchange secure proxy instead of exposing CoreLogic credentials
    auth_endpoint = 'https://idxexchange.com/internal-api/trestle_token.php?key=IDXEXCHANGE2026_CHANGE_THIS'

    response = requests.get(auth_endpoint, timeout=30)
    response.raise_for_status()



    # Parse the response to extract the token
    token = response.json().get('access_token')

    if token:

        # Define the headers with the token
        headers = {
            'Authorization': f'Bearer {token}'
        }

        # Define the parameters for the API request
        params = {
            '$select': 'OriginalListPrice, ListingKey,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,BuyerOfficeName,CoListOfficeName,ListAgentFullName,CoListAgentFirstName,CoListAgentLastName,BuyerAgentMlsId,BuyerAgentFirstName,BuyerAgentLastName,FireplacesTotal,AssociationFeeFrequency,AboveGradeFinishedArea,ListingKeyNumeric,MLSAreaMajor,TaxAnnualAmount,CountyOrParish,PropertyType,MlsStatus,ElementarySchool,ListAgentFirstName,AttachedGarageYN,ParkingTotal,BuilderName,PropertySubType,LotSizeAcres,SubdivisionName,BuyerOfficeAOR,YearBuilt,DaysOnMarket,StreetNumberNumeric,LivingArea,ListingId,BathroomsTotalInteger,City,TaxYear,BuildingAreaTotal,BedroomsTotal,ContractStatusChangeDate,Longitude,ElementarySchoolDistrict,CoBuyerAgentFirstName,PurchaseContractDate,ListingContractDate,BelowGradeFinishedArea,BusinessType,Latitude,ListPrice,StateOrProvince,CoveredSpaces,MiddleOrJuniorSchool,FireplaceYN,Stories,HighSchool,Levels,ListAgentLastName,CloseDate,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress',
      
            '$filter': f"ListingContractDate ge {datetime(file_year, file_month, 1).isoformat(timespec='milliseconds')}Z and ListingContractDate lt {datetime(file_year_end, file_month_end, 1).isoformat(timespec='milliseconds')}Z",


            '$top': 1000  # Extracting up to 1000 observations
        }

        # Send a GET request to the API endpoint with the token and parameters
        total_records = 0
        if file_month < 10:
            csv_file = f"CRMLSListing{file_year}0{file_month}.csv"
        else:
            csv_file = f"CRMLSListing{file_year}{file_month}.csv"

        with open(csv_file, mode='w', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=['OriginalListPrice','ListingKey', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'PropertyType', 'MlsStatus', 'ElementarySchool', 'ListAgentFirstName', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket', 'StreetNumberNumeric', 'LivingArea', 'ListingId', 'BathroomsTotalInteger', 'City',  'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate', 'Longitude', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'PurchaseContractDate', 'ListingContractDate', 'BelowGradeFinishedArea', 'BusinessType', 'Latitude', 'ListPrice', 'StateOrProvince', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'ListAgentLastName', 'CloseDate', 'LotSizeDimensions', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'BuyerOfficeName', 'AssociationFee', 'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict', 'UnparsedAddress'])
            writer.writeheader()

            while True:
                response = requests.get(url, params=params, headers=headers)
                if response.status_code == 200:
                    data = response.json()
                    observations = data.get('value', [])
                    for observation in observations:
                        writer.writerow({
                          'OriginalListPrice': observation.get('OriginalListPrice', ''),
                            'ListingKey': observation.get('ListingKey', ''),            
                            'CloseDate': observation.get('CloseDate', ''),
                            'ClosePrice': observation.get('ClosePrice', ''),
                            'ListAgentFirstName': observation.get('ListAgentFirstName', ''),
                            'ListAgentLastName': observation.get('ListAgentLastName', ''),
                            'Latitude': observation.get('Latitude', ''),
                            'Longitude': observation.get('Longitude', ''),
                            'UnparsedAddress': observation.get('UnparsedAddress', ''),
                            'PropertyType': observation.get('PropertyType', ''),
                            'LivingArea': observation.get('LivingArea', ''),
                            'ListPrice': observation.get('ListPrice', ''),
                            'DaysOnMarket': observation.get('DaysOnMarket', ''),
                            'ListOfficeName': observation.get('ListOfficeName', ''),
                            'BuyerOfficeName': observation.get('BuyerOfficeName', ''),
                            'CoListOfficeName': observation.get('CoListOfficeName', ''),
                            'ListAgentFullName': observation.get('ListAgentFullName', ''),
                            'CoListAgentFirstName': observation.get('CoListAgentFirstName', ''),
                            'CoListAgentLastName': observation.get('CoListAgentLastName', ''),
                            'BuyerAgentMlsId': observation.get('BuyerAgentMlsId', ''),
                            'BuyerAgentFirstName': observation.get('BuyerAgentFirstName', ''),
                            'BuyerAgentLastName': observation.get('BuyerAgentLastName', ''),
                            'FireplacesTotal': observation.get('FireplacesTotal', ''),
                            'AssociationFeeFrequency': observation.get('AssociationFeeFrequency', ''),
                            'AboveGradeFinishedArea': observation.get('AboveGradeFinishedArea', ''),
                            'ListingKeyNumeric': observation.get('ListingKeyNumeric', ''),
                            'MLSAreaMajor': observation.get('MLSAreaMajor', ''),
                            'TaxAnnualAmount': observation.get('TaxAnnualAmount', ''),
                            'CountyOrParish': observation.get('CountyOrParish', ''),
                            'MlsStatus': observation.get('MlsStatus', ''),
                            'ElementarySchool': observation.get('ElementarySchool', ''),
                            'ListAgentFirstName': observation.get('ListAgentFirstName', ''),
                            'AttachedGarageYN': observation.get('AttachedGarageYN', ''),
                            'ParkingTotal': observation.get('ParkingTotal', ''),
                            'BuilderName': observation.get('BuilderName', ''),
                            'PropertySubType': observation.get('PropertySubType', ''),
                            'LotSizeAcres': observation.get('LotSizeAcres', ''),
                            'SubdivisionName': observation.get('SubdivisionName', ''),
                            'BuyerOfficeAOR': observation.get('BuyerOfficeAOR', ''),
                            'YearBuilt': observation.get('YearBuilt', ''),
                            'DaysOnMarket': observation.get('DaysOnMarket', ''),                        
                            'StreetNumberNumeric': observation.get('StreetNumberNumeric', ''),
                            'LivingArea': observation.get('LivingArea', ''),
                            'ListingId': observation.get('ListingId', ''),
                            'BathroomsTotalInteger': observation.get('BathroomsTotalInteger', ''),
                            'City': observation.get('City', ''),
                            'TaxYear': observation.get('TaxYear', ''),
                            'BuildingAreaTotal': observation.get('BuildingAreaTotal', ''),
                            'BedroomsTotal': observation.get('BedroomsTotal', ''),
                            'ContractStatusChangeDate': observation.get('ContractStatusChangeDate', ''),
                            'Longitude': observation.get('Longitude', ''),
                            'ElementarySchoolDistrict': observation.get('ElementarySchoolDistrict', ''),
                            'CoBuyerAgentFirstName': observation.get('CoBuyerAgentFirstName', ''),
                            'PurchaseContractDate': observation.get('PurchaseContractDate', ''),
                            'ListingContractDate': observation.get('ListingContractDate', ''),
                            'BelowGradeFinishedArea': observation.get('BelowGradeFinishedArea', ''),
                            'BusinessType': observation.get('BusinessType', ''),
                            'Latitude': observation.get('Latitude', ''),
                            'ListPrice': observation.get('ListPrice', ''),
                            'StateOrProvince': observation.get('StateOrProvince', ''),
                            'CoveredSpaces': observation.get('CoveredSpaces', ''),
                            'MiddleOrJuniorSchool': observation.get('MiddleOrJuniorSchool', ''),
                            'FireplaceYN': observation.get('FireplaceYN', ''),
                            'Stories': observation.get('Stories', ''),
                            'HighSchool': observation.get('HighSchool', ''),
                            'Levels': observation.get('Levels', ''),
                            'ListAgentLastName': observation.get('ListAgentLastName', ''),
                            'CloseDate': observation.get('CloseDate', ''),
                            'LotSizeDimensions': observation.get('LotSizeDimensions', ''),
                            'LotSizeArea': observation.get('LotSizeArea', ''),
                            'MainLevelBedrooms': observation.get('MainLevelBedrooms', ''),
                            'NewConstructionYN': observation.get('NewConstructionYN', ''),
                            'GarageSpaces': observation.get('GarageSpaces', ''),
                            'HighSchoolDistrict': observation.get('HighSchoolDistrict', ''),
                            'PostalCode': observation.get('PostalCode', ''),
                            'BuyerOfficeName': observation.get('BuyerOfficeName', ''),
                            'AssociationFee': observation.get('AssociationFee', ''),
                            'LotSizeSquareFeet': observation.get('LotSizeSquareFeet', ''),
                            'MiddleOrJuniorSchoolDistrict': observation.get('MiddleOrJuniorSchoolDistrict', ''),
                            'UnparsedAddress': observation.get('UnparsedAddress', '')
                        })
                        total_records += 1

                    # Check if there are more records to fetch
                    if '@odata.nextLink' in data:
                        next_link = data['@odata.nextLink']
                        params = None  # Clear params to avoid appending to the existing query string
                        url = next_link
                    else:
                        break
                else:
                    print(f"Error: {response.status_code}")
                    print(f"Error Message: {response.text}")
                    break

        print(f"Total {total_records} records exported to {csv_file}")
    else:
        print("Error retrieving token: access_token not found")


In [5]:
# The following parameters retrieve the CRMLSListing and CRMLSSold files from (January 2024) to (May 2026), a total of (58) files.

first_year = 2024
first_month = 1
last_year = 2026
last_month = 5

for year in range(first_year, last_year+1):
    year_start_month = 1
    year_end_month = 12
    
    if year == first_year:
        year_start_month = first_month
    elif year == last_year:
        year_end_month = last_month
    
    for month in range(year_start_month, year_end_month+1):
        crmls_sold(year, month)
        crmls_listed(year, month)


Total 17847 records exported to CRMLSSold202401.csv
Total 23418 records exported to CRMLSListing202401.csv
Total 19782 records exported to CRMLSSold202402.csv
Total 22429 records exported to CRMLSListing202402.csv
Total 23077 records exported to CRMLSSold202403.csv
Total 25007 records exported to CRMLSListing202403.csv
Total 24761 records exported to CRMLSSold202404.csv
Total 27315 records exported to CRMLSListing202404.csv
Total 26776 records exported to CRMLSSold202405.csv
Total 28057 records exported to CRMLSListing202405.csv
Total 24165 records exported to CRMLSSold202406.csv
Total 25558 records exported to CRMLSListing202406.csv
Total 26110 records exported to CRMLSSold202407.csv
Total 25591 records exported to CRMLSListing202407.csv
Total 25201 records exported to CRMLSSold202408.csv
Total 24927 records exported to CRMLSListing202408.csv
Total 22220 records exported to CRMLSSold202409.csv
Total 23471 records exported to CRMLSListing202409.csv
Total 24076 records exported to CRMLS